# Notebook 1 - Exploratory Data Analysis

just exploring the movielens dataset before doing anything else

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')   # keep output clean
sns.set_theme(style='whitegrid')    # nice background for all plots
plt.rcParams['figure.figsize'] = (10, 5)  # default size for every chart

print('libraries loaded')

## Load Data

loading ratings and movies csv and merging them on movieId

In [ ]:
# load the csv files
ratings = pd.read_csv('data/ratings.csv', nrows=100000, dtype={
    'userId':  'int32',
    'movieId': 'int32',
    'rating':  'float32'
})
movies  = pd.read_csv('data/movies.csv', dtype={
    'movieId': 'int32'
})

# merge both
df = pd.merge(ratings, movies, on='movieId', how='left')

print('── ratings.csv : first 5 rows ──')
print(ratings.head())

print('\n── movies.csv : first 5 rows ──')
print(movies.head())

print('\n── merged dataframe : first 5 rows ──')
print(df.head())

## Basic Info

checking shape, dtypes and missing values

In [ ]:
# shape
print(f'ratings.csv  → {ratings.shape[0]:,} rows  × {ratings.shape[1]} columns')
print(f'movies.csv   → {movies.shape[0]:,} rows  × {movies.shape[1]} columns')
print(f'merged df    → {df.shape[0]:,} rows  × {df.shape[1]} columns')

print('\n── Data types ──')
print(df.dtypes)

print('\n── Missing values ──')
print(df.isnull().sum())

print('\n── Basic statistics for ratings ──')
print(df['rating'].describe())

## Rating Distribution

just checking how people rate movies, most give 3 or 4 stars

In [ ]:
# plot it
plt.figure(figsize=(10, 5))
sns.histplot(df['rating'], bins=10, kde=True, color='steelblue')
plt.title('Rating Distribution — How Users Rate Movies', fontsize=14)
plt.xlabel('Rating (0.5 to 5.0)', fontsize=12)
plt.ylabel('Number of Ratings', fontsize=12)
plt.xticks([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5])
plt.tight_layout()
plt.savefig('reports/rating_distribution.png', dpi=80)
plt.show()
plt.close('all')
print('Plot saved to reports/rating_distribution.png')

## Top 10 Most Rated Movies

movies with the most number of ratings, not the highest rated

In [ ]:
# count ratings per movie
most_rated = (df.groupby('title')['rating']
                .count()
                .sort_values(ascending=False)
                .head(10))

plt.figure(figsize=(12, 5))
sns.barplot(x=most_rated.values, y=most_rated.index, palette='Blues_r')
plt.title('Top 10 Most Rated Movies', fontsize=14)
plt.xlabel('Number of Ratings', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.tight_layout()
plt.savefig('reports/top10_most_rated.png', dpi=80)
plt.show()
plt.close('all')

print('\nTop 10 Most Rated Movies:')
print(most_rated.to_frame('rating_count'))

## Top 10 Highest-Rated Movies

filtering to min 50 ratings otherwise random obscure movies end up at the top

In [ ]:
# get count and mean per movie
movie_stats = (df.groupby('title')['rating']
                 .agg(['count', 'mean'])
                 .reset_index())
movie_stats.columns = ['title', 'num_ratings', 'avg_rating']

# only movies with at least 50 ratings
qualified = movie_stats[movie_stats['num_ratings'] >= 50]
top_rated = qualified.sort_values('avg_rating', ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x='avg_rating', y='title', data=top_rated, palette='Greens_r')
plt.title('Top 10 Highest-Rated Movies (min 50 ratings)', fontsize=14)
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.xlim(3.5, 5.0)   # zoom in so differences are visible
plt.tight_layout()
plt.savefig('reports/top10_highest_rated.png', dpi=80)
plt.show()
plt.close('all')

print('\nTop 10 Highest Rated Movies (min 50 ratings):')
print(top_rated[['title', 'avg_rating', 'num_ratings']].to_string(index=False))

## Genre Frequency

genres are separated by | so need to split them first

In [ ]:
# split genres and count
all_genres = (movies['genres']
              .str.split('|')           # split on pipe: 'Action|Drama' → ['Action','Drama']
              .explode()               # one genre per row
              .value_counts()          # count occurrences
              .drop('(no genres listed)', errors='ignore'))  # remove unknown

plt.figure(figsize=(14, 6))
sns.barplot(x=all_genres.values, y=all_genres.index, palette='viridis')
plt.title('Genre Frequency Count — How Often Each Genre Appears', fontsize=14)
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.tight_layout()
plt.savefig('reports/genre_frequency.png', dpi=80)
plt.show()
plt.close('all')

print('\nGenre Counts:')
print(all_genres)

## Average Rating Per Genre

which genres get better ratings on average

In [ ]:
# explode genres and group
avg_by_genre = (df[['genres', 'rating']]
                .assign(genre_split=df['genres'].str.split('|'))
                .explode('genre_split')
                .query('genre_split != "(no genres listed)"')
                .groupby('genre_split')['rating']
                .mean()
                .sort_values(ascending=False))

plt.figure(figsize=(12, 7))
sns.barplot(x=avg_by_genre.values, y=avg_by_genre.index, palette='coolwarm')
plt.title('Average Rating Per Genre', fontsize=14)
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.xlim(2.5, 4.5)
plt.tight_layout()
plt.savefig('reports/avg_rating_per_genre.png', dpi=80)
plt.show()
plt.close('all')

print('\nAverage Rating Per Genre:')
print(avg_by_genre.round(3))

## Ratings Per User

some users rate a lot of movies, some rate very few

In [ ]:
# count per user
ratings_per_user = ratings.groupby('userId')['rating'].count()

plt.figure(figsize=(10, 5))
sns.histplot(ratings_per_user, bins=50, color='coral', kde=True)
plt.title('Number of Ratings Per User — User Activity Distribution', fontsize=14)
plt.xlabel('Number of Ratings Given by User', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)
plt.tight_layout()
plt.savefig('reports/ratings_per_user.png', dpi=80)
plt.show()
plt.close('all')

print(f'Most active user rated {ratings_per_user.max()} movies')
print(f'Least active user rated {ratings_per_user.min()} movies')
print(f'Median ratings per user: {ratings_per_user.median():.0f}')

print('\ndone')